# Módulo 5 — Análise Exploratória, Indicadores e Fatores Associados

In [22]:
## Bibliotecas principais do Módulo 5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [23]:
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

ARQUIVO_BRUTO = RAIZ / "dados_brutos" / "acidentes2025.csv"
ARQUIVO_BASE_ANALITICA = RAIZ / "dados_tratados" / "base_analitica_prf_2025.csv"
ARQUIVO_BASE_MODELAVEL = RAIZ / "dados_tratados" / "base_modelavel_prf_2025.csv"
ARQUIVO_DICIONARIO = RAIZ / "dados_tratados" / "dicionario_variaveis_modulo4.csv"
ARQUIVO_DECISOES = RAIZ / "logs" / "decisoes_tratamento_modulo4.md"
ARQUIVO_README = RAIZ / "docs" / "README.md"

SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [24]:
df = pd.read_csv(ARQUIVO_BASE_ANALITICA,
                    sep=SEPARADOR,
                    encoding=ENCODING_SAIDA)
df["acidente_fatal"] = (df["mortos"] >= 1).astype(int)

total     = len(df)
fatais    = df["acidente_fatal"].sum()
pct_fatal = fatais / total
mortos    = df["mortos"].sum()

In [25]:
# Tipos de dados e memória utilizada
df.info(memory_usage="deep")

resumo_tipos = (
 df.dtypes.astype(str)
 .value_counts()
 .rename_axis("tipo")
 .reset_index(name="qtd_colunas")
)
display(resumo_tipos)

<class 'pandas.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      72529 non-null  int64  
 1   data_inversa            72529 non-null  str    
 2   dia_semana              72529 non-null  str    
 3   horario                 72529 non-null  str    
 4   uf                      72529 non-null  str    
 5   br                      72529 non-null  int64  
 6   km                      72529 non-null  float64
 7   municipio               72529 non-null  str    
 8   causa_acidente          72529 non-null  str    
 9   tipo_acidente           72529 non-null  str    
 10  classificacao_acidente  72529 non-null  str    
 11  fase_dia                72529 non-null  str    
 12  sentido_via             72529 non-null  str    
 13  condicao_metereologica  72529 non-null  str    
 14  tipo_pista              72529 non-null  str    
 

,tipo,qtd_colunas
0,str,23
1,int64,20
2,float64,1


In [26]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana,hora,turno,faixa_horaria,acidente_fatal,total_vitimas,acidente_grave,indice_gravidade,br_formatada,chave_localidade
0,652493,2025-01-01,QUARTA-FEIRA,06:20:00,SP,116,225.0,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,"['RETA', 'DECLIVE']",SIM,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0,6,MANHA,06H-08H,0,1,0,1,BR-116,SP_GUARULHOS_BR-116
1,652519,2025-01-01,QUARTA-FEIRA,07:50:00,CE,116,546.2,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,COM VÍTIMAS FATAIS,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,['RETA'],NÃO,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0,7,MANHA,06H-08H,1,2,1,4,BR-116,CE_PENAFORTE_BR-116
2,652522,2025-01-01,QUARTA-FEIRA,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,"['RETA', 'ACLIVE']",SIM,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0,8,MANHA,06H-08H,0,3,0,3,BR-369,PR_CORNELIO PROCOPIO_BR-369
3,652544,2025-01-01,QUARTA-FEIRA,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,['RETA'],NÃO,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0,11,MANHA,09H-11H,0,1,0,1,BR-116,PR_CAMPINA GRANDE DO SUL_BR-116
4,652549,2025-01-01,QUARTA-FEIRA,09:30:00,MG,251,471.0,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,"['CURVA', 'DECLIVE']",NÃO,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0,9,MANHA,09H-11H,0,2,1,3,BR-251,MG_FRANCISCO SA_BR-251


In [42]:
def indicadores_por_categoria(base, coluna, min_registros=30):
    df_calc = base.copy()

    df_calc["taxa_letalidade"] = df_calc["mortos"] / df_calc["pessoas"]

    tab = df_calc.groupby(coluna).agg(
    total_acidentes=("acidente_fatal","size"),
    total_mortos=("mortos", "sum"),
    total_feridos=("feridos", "sum"),
    total_acidentes_fatais=("acidente_fatal","sum"),
    taxa_fatal=("acidente_fatal","mean"),
    taxa_letalidade=("taxa_letalidade", "mean")
    ).reset_index()
    tab = tab[tab["total_acidentes"] >= min_registros]
    return tab.sort_values("taxa_fatal", ascending=False)

In [43]:
display(indicadores_por_categoria(df, "tipo_acidente",min_registros=30).head(10))

,tipo_acidente,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
1,ATROPELAMENTO DE PEDESTRE,3057,919,2846,902,0.295061,0.122671
4,COLISÃO FRONTAL,4739,1863,7596,1396,0.294577,0.111277
6,COLISÃO LATERAL SENTIDO OPOSTO,2152,255,2824,212,0.098513,0.030106
11,EVENTOS ATÍPICOS,287,24,293,23,0.080139,0.038993
0,ATROPELAMENTO DE ANIMAL,1133,74,1267,68,0.060018,0.036746
14,SAÍDA DE LEITO CARROÇÁVEL,10209,700,11638,605,0.059261,0.038951
3,COLISÃO COM OBJETO,5109,323,4952,297,0.058133,0.038295
2,CAPOTAMENTO,1373,74,1827,63,0.045885,0.026843
7,COLISÃO TRANSVERSAL,9306,481,12015,427,0.045884,0.017280
8,COLISÃO TRASEIRA,14360,683,16376,619,0.043106,0.015780


In [44]:
display(indicadores_por_categoria(df, "causa_acidente",min_registros=30).head(10))

,causa_acidente,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
60,SUICÍDIO (PRESUMIDO),190,111,98,106,0.557895,0.215705
45,PEDESTRE ANDAVA NA PISTA,606,252,465,250,0.412541,0.174752
27,ENTRADA INOPINADA DO PEDESTRE,667,209,582,203,0.304348,0.123260
63,TRANSITAR NA CONTRAMÃO,2475,961,3452,736,0.297374,0.111517
46,PEDESTRE CRUZAVA A PISTA FORA DA FAIXA,525,133,534,130,0.247619,0.099870
30,FALTA DE ACOSTAMENTO,60,11,58,11,0.183333,0.073611
44,PEDESTRE - INGESTÃO DE ÁLCOOL/ SUBSTÂNCIAS PSI...,150,28,163,27,0.180000,0.079333
66,ULTRAPASSAGEM INDEVIDA,1770,404,2760,302,0.170621,0.059957
65,TRANSTORNOS MENTAIS (EXCETO SUICIDIO),59,10,51,10,0.169492,0.070621
68,ÁREA URBANA SEM A PRESENÇA DE LOCAL APROPRIADO...,81,12,89,12,0.148148,0.067901


In [45]:
display(indicadores_por_categoria(df, "uf",min_registros=30))

,uf,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
9,MA,1262,281,1295,236,0.187005,0.090177
13,PA,1117,224,1191,193,0.172784,0.076644
21,RR,142,28,193,23,0.161972,0.060070
2,AM,138,26,149,19,0.137681,0.062207
1,AL,629,96,740,86,0.136725,0.069877
26,TO,677,102,784,83,0.122600,0.051406
5,CE,1302,171,1429,153,0.117512,0.051475
4,BA,4108,583,5023,476,0.115871,0.050810
0,AC,280,29,328,29,0.103571,0.048516
15,PE,3013,336,3348,302,0.100232,0.049092


In [47]:
display(indicadores_por_categoria(df, "br",min_registros=30).head(10))

,br,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
44,308,34,10,41,10,0.294118,0.107843
67,402,63,23,69,16,0.253968,0.128968
51,349,52,16,78,12,0.230769,0.078480
69,405,97,22,91,22,0.226804,0.128351
46,317,53,12,47,12,0.226415,0.130189
82,423,183,59,229,40,0.218579,0.104415
83,424,91,23,98,19,0.208791,0.121978
28,242,283,73,392,54,0.190813,0.072308
23,222,581,128,594,106,0.182444,0.087161
1,10,506,95,604,86,0.169960,0.074257


In [59]:
display(indicadores_por_categoria(df, "municipio",min_registros=30).head(10))

,municipio,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
1256,PESQUEIRA,30,13,48,11,0.366667,0.160556
796,ITAPECURU MIRIM,40,16,45,11,0.275000,0.132500
413,CAXIAS,67,23,75,18,0.268657,0.114925
1731,TIMON,52,16,48,13,0.250000,0.133013
485,CORRENTINA,54,16,79,13,0.240741,0.079111
513,CURITIBANOS,30,7,32,7,0.233333,0.087222
262,BURITIZEIRO,30,9,57,7,0.233333,0.089846
1299,PONTA PORA,30,7,39,7,0.233333,0.138889
218,BOM DESPACHO,35,9,36,8,0.228571,0.075782
1221,PAUDALHO,41,9,31,8,0.195122,0.117886


In [58]:
display(indicadores_por_categoria(df, "fase_dia",min_registros=30))

,fase_dia,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
0,AMANHECER,3447,472,3761,386,0.111981,0.053453
2,PLENA NOITE,24781,2892,27179,2522,0.101772,0.047112
1,ANOITECER,3926,288,4558,253,0.064442,0.024935
3,PLENO DIA,40375,2391,48052,2049,0.050749,0.021290


In [60]:
display(indicadores_por_categoria(df, "condicao_metereologica",min_registros=30).head(10))

,condicao_metereologica,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
5,NEVOEIRO/NEBLINA,553,74,617,60,0.108499,0.053802
3,IGNORADO,1000,115,1025,99,0.099000,0.044290
8,VENTO,104,10,117,9,0.086538,0.035256
1,CÉU CLARO,46375,3948,52641,3419,0.073725,0.032613
6,NUBLADO,11435,955,13176,830,0.072584,0.032400
0,CHUVA,6438,480,8192,402,0.062442,0.027288
2,GAROA/CHUVISCO,2422,175,2793,144,0.059455,0.027318
7,SOL,4201,286,4988,247,0.058796,0.025405


In [61]:
display(indicadores_por_categoria(df, "tipo_pista",min_registros=30))

,tipo_pista,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
2,SIMPLES,34733,4143,41939,3424,0.098581,0.042923
0,DUPLA,30782,1603,33864,1501,0.048762,0.022282
1,MÚLTIPLA,7014,297,7747,285,0.040633,0.018888


In [62]:
display(indicadores_por_categoria(df, "chave_localidade",min_registros=30).head(10))

,chave_localidade,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
1358,PE_PESQUEIRA_BR-232,30,13,48,11,0.366667,0.160556
678,MA_TIMON_BR-316,38,14,37,11,0.289474,0.155702
608,MA_CAXIAS_BR-316,62,22,69,17,0.274194,0.118817
2194,SC_CAMPOS NOVOS_BR-470,36,11,30,9,0.250000,0.094841
721,MG_BURITIZEIRO_BR-365,30,9,57,7,0.233333,0.089846
1015,MS_PONTA PORA_BR-463,30,7,39,7,0.233333,0.138889
2208,SC_CURITIBANOS_BR-470,30,7,32,7,0.233333,0.087222
715,MG_BOM DESPACHO_BR-262,35,9,36,8,0.228571,0.075782
166,BA_CORRENTINA_BR-349,45,13,75,10,0.222222,0.072169
579,MA_ACAILANDIA_BR-010,33,9,37,7,0.212121,0.084343


In [63]:
display(indicadores_por_categoria(df, "faixa_horaria",min_registros=30))

,faixa_horaria,total_acidentes,total_mortos,total_feridos,total_acidentes_fatais,taxa_fatal,taxa_letalidade
1,03H-05H,4948,764,5223,639,0.129143,0.063255
0,00H-02H,3959,506,3959,439,0.110887,0.055237
7,21H-23H,6988,789,7638,703,0.100601,0.046359
6,18H-20H,13473,1320,15480,1166,0.086543,0.037037
5,15H-17H,12624,819,15151,706,0.055925,0.023112
4,12H-14H,9678,615,11737,528,0.054557,0.022279
2,06H-08H,11517,693,13453,571,0.049579,0.022266
3,09H-11H,9342,537,10909,458,0.049026,0.020419


In [1]:
def indicadores_por_categoria_ordenado_por(base, coluna, min_registros=30, coluna_ordenacao="taxa_fatal"):
    df_calc = base.copy()

    df_calc["taxa_letalidade"] = df_calc["mortos"] / df_calc["pessoas"]

    tab = df_calc.groupby(coluna).agg(
    total_acidentes=("acidente_fatal","size"),
    total_mortos=("mortos", "sum"),
    total_feridos=("feridos", "sum"),
    total_acidentes_fatais=("acidente_fatal","sum"),
    taxa_fatal=("acidente_fatal","mean"),
    taxa_letalidade=("taxa_letalidade", "mean")
    ).reset_index()
    tab = tab[tab["total_acidentes"] >= min_registros]
    return tab.sort_values(coluna_ordenacao, ascending=False)

In [ ]:
def indicadores_por_categoria_ordenado_por(base, coluna, min_registros=30, coluna_ordenacao="taxa_fatal"):
    df_calc = base.copy()

    df_calc["taxa_letalidade"] = df_calc["mortos"] / df_calc["pessoas"]

    tab = df_calc.groupby(coluna).agg(
        total_acidentes=("acidente_fatal", "size"),
        total_mortos=("mortos", "sum"),
        total_feridos=("feridos", "sum"),
        total_acidentes_fatais=("acidente_fatal", "sum"),
        taxa_fatal=("acidente_fatal", "mean"),
        taxa_letalidade=("taxa_letalidade", "mean")
    ).reset_index()

    tab = tab[tab["total_acidentes"] >= min_registros]
    return tab.sort_values(coluna_ordenacao, ascending=False)